# ASD spectra: albedo, broadband albedo, and reflectance

Driver notebook for the [`asdspec`](../asdspec) package. Reads ASD FieldSpec binary files directly,
runs QC on the replicate measurements, and produces spectral albedo, broadband albedo integrated
against measured or modeled irradiance, and panel-referenced transect reflectance.

Method notes, including the integration-time and splice corrections and what the QC numbers mean,
are in the [README](../README.md). Edit the settings block in section 1 and run the notebook top to
bottom.

## 1. Setup

In [ ]:
# In Colab, install the package and skip the sys.path line:
#   !pip install -q git+https://github.com/<user>/<repo>.git
# Running from a clone, make the package importable instead:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import asdspec as asd
from asdspec import plots
from asdspec.irradiance import TRAPZ

pd.set_option("display.width", 200)
pd.set_option("display.max_rows", 400)
plt.rcParams.update({"figure.dpi": 110, "font.size": 9, "axes.grid": True,
                     "grid.alpha": 0.3, "figure.facecolor": "white"})
print(f"asdspec {asd.__version__}")

In [ ]:
# ----------------------------------------------------------------------
# Settings
# ----------------------------------------------------------------------
DATA_DIR = "../example_data"        # folder of ASD files, searched recursively
OUTPUT_DIR = "../output"

# Processing
SPLICE_MODE = "swir1_anchor"        # "swir1_anchor" | "vnir_anchor" | "none"
CORRECT_SWIR2 = False               # 1800 nm splice; see README
MANUAL_IT_FACTOR = None             # override the integration-time factor
Z_THRESHOLD = 3.5                   # outlier flag level
SNR_MIN = 5.0                       # 0 disables the data-driven noise mask

# Exclusions: block_id -> file indices to drop, e.g. {3: [7], 4: [0, 9]}
EXCLUDE = {}
AUTO_EXCLUDE_FLAGGED = False

# Irradiance weighting for broadband albedo
IRRADIANCE_SOURCE = "auto"          # "auto" | "measured" | "modeled"
MODELED_IRRADIANCE_FILE = None      # None looks for a *irrad*.csv near the data
MANUAL_SZA = None                   # degrees; or give the site below instead
SITE_LAT = None                     # e.g. 40.60
SITE_LON = None                     # e.g. -111.60, west negative
UTC_OFFSET_HOURS = None             # e.g. -6. ASD headers store local wall-clock time.

# Reflectance
PANEL_REFLECTANCE = 0.99            # flat placeholder; see README
PANEL_FILE = None                   # CSV: wavelength_nm, reflectance
REFLECTANCE_STEMS = []              # empty means auto-detect
# ----------------------------------------------------------------------

DATA_DIR, OUTPUT_DIR = Path(DATA_DIR), Path(OUTPUT_DIR)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# In Colab, upload a zip instead:
#   from google.colab import files, zipfile, io
#   up = files.upload(); DATA_DIR = Path('/content/asd_data'); DATA_DIR.mkdir(exist_ok=True)
#   for name, blob in up.items():
#       if name.endswith('.zip'):
#           zipfile.ZipFile(io.BytesIO(blob)).extractall(DATA_DIR)
#       else:
#           (DATA_DIR / name).write_bytes(blob)

data = asd.scan_folder(DATA_DIR)
WL = data.wavelength
print(f"{len(data)} spectra, {WL[0]:.0f}-{WL[-1]:.0f} nm at {WL[1] - WL[0]:.0f} nm")

## 2. Inventory and blocks

A block is a run of consecutive files sharing role, integration time, gain, offset and spectrum
type, with no long pause. That is the unit averaged over. Check this table against the field notes
before going on: one file stem often holds several optimizations, and a reflectance day is usually
opening panel, transect, closing panel under a single stem.

In [ ]:
display(data.inventory.groupby(["stem", "type", "it_ms", "role"], as_index=False)
        .agg(n=("file", "size"), first=("time", "min"), last=("time", "max"),
             swir_vis=("swir_vis_ratio", "mean"),
             swir1_gain=("swir1_gain", "first"), swir1_offset=("swir1_offset", "first")))

blocks = asd.find_blocks(data)
block_table = asd.block_table(blocks)
display(block_table)

reflectance_stems = REFLECTANCE_STEMS or asd.detect_reflectance_stems(blocks)
pairs, orphans = asd.auto_pairs(blocks, skip_stems=reflectance_stems)
# override by hand if a set is split across stems, e.g. pairs = [(0, 1), (2, 4)]

if reflectance_stems:
    print("reflectance days, handled in section 5:", reflectance_stems)
print("albedo pairs (reference block, target block):", pairs)
for bid in orphans:
    b = blocks[bid]
    print(f"   unpaired: block {bid}  {b['stem']}  {b['role']}  n={b['n']}  "
          f"{b['t_start']:%H:%M:%S}  it={b['it_ms']} ms")

## 3. QC

Every replicate is scored on overall level and on spectral shape, both as modified z-scores built
on the median absolute deviation. Nothing is dropped automatically: review the plots, then record
exclusions in the `EXCLUDE` dict in section 1 and re-run. Keeping exclusions in the notebook rather
than deleting files keeps the processing reproducible.

In [ ]:
qc = asd.qc_all(data, blocks, EXCLUDE, z_threshold=Z_THRESHOLD)

if AUTO_EXCLUDE_FLAGGED:
    for bid, scores in qc.items():
        auto = scores.loc[scores["flagged"], "index"].tolist()
        if auto:
            EXCLUDE[bid] = sorted(set(EXCLUDE.get(bid, [])) | set(auto))

for bid, block in blocks.items():
    scores = qc[bid]
    plots.plot_block_qc(data, block, scores, Z_THRESHOLD)
    plt.show()
    flagged = scores.loc[scores["flagged"], "index"].tolist()
    dropped = EXCLUDE.get(bid, [])
    print(f"   flagged: {flagged or 'none'}    excluded: {dropped or 'none'}")

## 4. Spectral albedo

Target set over reference set, with the integration-time correction, then the splice correction,
then masking.

The `vnir_splice_factor` column is the number to read. After a correct integration-time correction
it sits near 0.985. A value well outside 0.96 to 1.01 flags a pair where the instrument was
re-optimized between reference and target, so the integration-time factor is suspect. Section 4b
shows the splice region before and after correction.

In [ ]:
results = [asd.compute_albedo(data, blocks[r], blocks[t], exclude=EXCLUDE,
                              manual_it_factor=MANUAL_IT_FACTOR, splice_mode=SPLICE_MODE,
                              correct_swir2=CORRECT_SWIR2, snr_min=SNR_MIN)
           for r, t in pairs]

if results:
    qc_table = asd.splice_qc_table(results)
    display(qc_table)
    lo, hi = asd.SPLICE_FACTOR_OK
    suspect = qc_table[(qc_table["vnir_splice_factor"] < lo)
                       | (qc_table["vnir_splice_factor"] > hi)]
    if len(suspect):
        print(f"\nSplice factor outside {lo} to {hi}. Check integration times and field notes:")
        display(suspect[["set", "it_factor", "raw_splice_step_pct", "vnir_splice_factor"]])

    plots.plot_albedo(results, SPLICE_MODE)
    plt.show()
else:
    print("No albedo pairs. If this is a reflectance-only day, go to section 5.")

In [ ]:
# 4b. Splice diagnostic. A clean result has no jump at the red line on the right.
if results:
    plots.plot_splice_diagnostic(results)
    plt.show()

## 5. Broadband albedo

Irradiance-weighted mean albedo. The weighting spectrum must be in physical units; raw uplooking DN
will not do. Measured irradiance is used when the day has a calibrated set, otherwise a clear-sky
table indexed by solar zenith angle. Give either `MANUAL_SZA`, or the site coordinates and UTC
offset so the angle is computed per set from the file timestamps.

Expect a systematic offset of a few hundredths between measured and modeled weighting, since a
clear-sky model cannot know the day's cloud and diffuse fraction. The choice of zenith angle column
matters far less than that.

In [ ]:
model = None
model_path = MODELED_IRRADIANCE_FILE
if model_path is None:
    candidates = [p for p in list(DATA_DIR.rglob("*.csv")) + list(Path.cwd().glob("*.csv"))
                  if "irrad" in p.name.lower()]
    model_path = candidates[0] if candidates else None
if model_path is not None:
    model = asd.ModeledIrradiance.from_csv(model_path, WL)
    print(model.describe())

irradiance_blocks = [b for b in blocks.values()
                     if b["type"] == "IRRADIANCE" and b["role"] == "reference"]
E_measured = None
if irradiance_blocks:
    block = irradiance_blocks[0]
    raw, kept = data.mean(block, EXCLUDE.get(block["block_id"], ()))
    E_measured, _ = asd.mask_fill(WL, raw, windows=[(2400, 2500)])
    E_measured = np.clip(E_measured, 0, None)
    print(f"measured irradiance: block {block['block_id']} ({block['stem']}, n={len(kept)}, "
          f"{block['t_start']:%H:%M:%S}), {TRAPZ(E_measured, WL):.0f} W m-2 over 350-2500 nm")
else:
    print("no calibrated irradiance set on this day")


def zenith_for(when):
    if MANUAL_SZA is not None:
        return float(MANUAL_SZA)
    if when is None or None in (SITE_LAT, SITE_LON, UTC_OFFSET_HOURS):
        return None
    return asd.solar_zenith(when, SITE_LAT, SITE_LON, UTC_OFFSET_HOURS)


def weighting_for(result):
    if IRRADIANCE_SOURCE in ("auto", "measured") and E_measured is not None:
        return dict(E=E_measured, source="measured", sza=np.nan, column="", clamped=False)
    if IRRADIANCE_SOURCE == "measured" or model is None:
        return dict(E=None, source="unavailable", sza=np.nan, column="", clamped=False)
    sza = zenith_for(result["time"])
    E, angle, clamped = model.for_sza(sza)
    if E is None:
        return dict(E=None, source="unavailable", sza=np.nan, column="", clamped=False)
    return dict(E=E, source="modeled", sza=round(sza, 1), column=f"Z{angle:g}", clamped=clamped)


if E_measured is not None and model is not None and results:
    sza = zenith_for(results[0]["time"])
    E_model, angle, _ = model.for_sza(sza if sza is not None else np.median(model.angles))
    plots.plot_irradiance_comparison(WL, E_measured, E_model, f"modeled Z{angle:g}")
    plt.show()
    for lo, hi in [(350, 700), (700, 1000), (1000, 1500), (1500, 2500)]:
        m = (WL >= lo) & (WL <= hi)
        a = TRAPZ(E_measured[m], WL[m]) / TRAPZ(E_measured, WL)
        b = TRAPZ(E_model[m], WL[m]) / TRAPZ(E_model, WL)
        print(f"   {lo}-{hi} nm: measured {100 * a:5.1f} %, modeled {100 * b:5.1f} % "
              f"({100 * (b - a):+.1f} points)")

In [ ]:
rows, n_clamped = [], 0
for result in results:
    weight = weighting_for(result)
    if weight["E"] is None:
        print(f"{result['label']}: no irradiance weighting available, skipped")
        continue
    n_clamped += bool(weight["clamped"])
    bands = asd.broadband_albedo(WL, result["albedo"], weight["E"])
    rows.append(dict(set=result["label"],
                     time=result["time"].strftime("%Y-%m-%d %H:%M") if result["time"] else "",
                     weighting=weight["source"], sza=weight["sza"], column=weight["column"],
                     clamped=weight["clamped"],
                     splice_factor=round(result["splice1_factor"], 4),
                     **{k: round(v, 4) for k, v in bands.items()}))

broadband_table = pd.DataFrame(rows) if rows else None
if broadband_table is not None:
    display(broadband_table)
    if n_clamped:
        print(f"\n{n_clamped} set(s) fall outside the zenith-angle range of the table "
              f"({min(model.angles):g} to {max(model.angles):g} deg) and were clamped to the "
              f"nearest column. Near solar noon in spring at mid-latitude this is expected.")

## 6. Reflectance

The white reference set is averaged; transect points are not, since the variability along the
transect is the signal. With a narrow foreoptic under natural illumination this is an HDRF, not a
bi-hemispherical albedo, and over snow and ice the two are not interchangeable. Keep them labelled
separately.

In [ ]:
if PANEL_FILE:
    panel_curve = pd.read_csv(PANEL_FILE)
    panel_reflectance = np.interp(WL, panel_curve["wavelength_nm"], panel_curve["reflectance"])
else:
    panel_reflectance = PANEL_REFLECTANCE

reflectance_results = []
for stem in reflectance_stems:
    refs = [b for b in blocks.values() if b["stem"] == stem and b["role"] == "reference"]
    targets = sorted([b for b in blocks.values() if b["stem"] == stem and b["role"] == "target"],
                     key=lambda b: b["t_start"])
    print(f"\n{stem}: {len(refs)} panel block(s), {len(targets)} transect block(s)")

    drift = asd.panel_drift(data, blocks, stem)
    if drift is not None:
        ratio, same_settings, minutes = drift
        summary = "  ".join(f"{w} nm {ratio[int(w - WL[0])]:.3f}" for w in (500, 900, 1500, 2000))
        print(f"  panel drift over {minutes:.1f} min (closing / opening): {summary}")
        if not same_settings:
            print("  the panels were taken with different SWIR gain or offset, so the SWIR "
                  "figures above are a settings change rather than drift")

    for target in targets:
        panel, reason = asd.choose_panel(blocks, stem, target)
        result = asd.compute_reflectance(data, panel, target, panel_reflectance, EXCLUDE,
                                         MANUAL_IT_FACTOR, SPLICE_MODE, CORRECT_SWIR2)
        reflectance_results.append(result)
        print(f"  transect block {target['block_id']}: {len(result['labels'])} points, "
              f"panel = block {panel['block_id']} ({reason}, n={result['n_panel']})")
        for note in result["notes"]:
            print("      " + note)
        plots.plot_reflectance(result, asd.DEFAULT_MASK)
        plt.show()

if not reflectance_stems:
    print("No reflectance day detected. Set REFLECTANCE_STEMS if one of the stems is a "
          "panel-plus-transect day.")
    print("Stems present:", sorted(data.inventory["stem"].unique()))

## 7. Export

In [ ]:
data.inventory.drop(columns=["path"]).to_csv(OUTPUT_DIR / "inventory.csv", index=False)
block_table.to_csv(OUTPUT_DIR / "blocks.csv", index=False)
pd.concat([s.assign(block=bid) for bid, s in qc.items()], ignore_index=True) \
    .to_csv(OUTPUT_DIR / "qc_scores.csv", index=False)

if results:
    spectral = pd.DataFrame({"wavelength_nm": WL})
    for result in results:
        spectral[result["label"]] = np.where(result["masked"], np.nan, result["albedo"])
    spectral.to_csv(OUTPUT_DIR / "spectral_albedo.csv", index=False)
    asd.splice_qc_table(results).to_csv(OUTPUT_DIR / "albedo_qc.csv", index=False)
if broadband_table is not None:
    broadband_table.to_csv(OUTPUT_DIR / "broadband_albedo.csv", index=False)
for result in reflectance_results:
    frame = pd.DataFrame(result["reflectance"].T,
                         columns=[f"pt_{i:03d}" for i in result["labels"]])
    frame.insert(0, "wavelength_nm", WL)
    frame.to_csv(OUTPUT_DIR / f"reflectance_{result['label'].replace(' ', '_')}.csv", index=False)

print("wrote:", *sorted(p.name for p in OUTPUT_DIR.iterdir()), sep="\n  ")